# E9 GNN Navigation

Author: Arush Arora

## Introduction

This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the shortest-distance paths of the graph before expecting it to serve the LLM with **multiplicative** GREPs for navigation tasks, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)

The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)

The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

### Sparse Graph Transformer

The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

### Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

## Setup

In [1]:
# %env CUDA_VISIBLE_DEVICES=0
%load_ext autoreload
%autoreload 2

In [2]:
# Import modules.
import gc
import copy
import wandb
import torch
import random
import bisect
import pickle
import sympy as sp
import networkx as nx

from typing import Union

from torch import nn
from torch_geometric.data import Data
from torch.distributions import Cauchy
from torch.nn.utils import clip_grad_norm_
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.utils import to_dense_adj, to_networkx

from prism.models.gt import GraphTransformer, SemanticGraphTransformer
from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.models.gcn import GCN
from prism.data import data, utils

In [3]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e9-gnn-navigation'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage` ('edge_detection' / 'path_navigation').

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit='return_previous',
    )

In [4]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 3, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [5]:
# Standard options.
ex_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
eval_path = ex_path # '../data_store/old/eval/e6_transferability'
save_path = '../data/pickle/e6_eval_graphs.pkl'
device = 'cuda'

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(ex_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_025.html


## Experiments

### §1 Pretraining a GNN to Classify Edge Existence

We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify whether an edge exists in the graph or not. Such a model will serve as a backbone pretrained model for fine-tuning on reporting shortest paths. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ \Vert\ \mathbf{h}_j\ \Vert\ \mathbf{h}_i \odot \mathbf{h}_j\ \Vert\ |\mathbf{h}_i - \mathbf{h}_j\|\big] \in [0, 1]$$

#### Model Definitions

We first define the models.

In [8]:
# Instantiate a GNN. `model_type` / `model_hparams` are exposed at module scope so
# init_wandb can log the GNN config; create_gnn writes model_hparams as it builds.
def create_gnn(model_type: str):
    global model_hparams
    if model_type == 'gt':
        model_hparams = dict(
            num_layers=3,
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            heads=8,
            num_samples=320,
            dropout=0.1,
            k_pe=3,
            k_gt=2,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = GraphTransformer(**model_hparams)
        gnn.out_features = gnn.d_model
    else:
        model_hparams = dict(
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            num_samples=320,
            dropout=0.1,
            k=3,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = RandomGNNPositionalEncodings(**model_hparams)
        gnn.out_features = gnn.output_projection.out_features
    return gnn


model_type = 'gt'
gnn = create_gnn(model_type)

In [9]:
# Define a class for edge detection and instantiate it.
class GNNEdgeDetector(nn.Module):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNEdgeDetector, self).__init__()
        self.gnn = gnn
        shape = self.gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(4 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, shape))

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        hi, hj = self.cached_pe[node1], self.cached_pe[node2]
        return self.classifier(torch.cat((hi, hj, hi * hj, abs(hi - hj)), dim=0))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
detector = GNNEdgeDetector(gnn)

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the graph adjacency given a scene graph PyTorch `Data` object.

In [10]:
# Prepare a graph from the data to be used in the GNN.
load_ex_graph = False

if load_ex_graph:
    with open(save_path, 'rb') as file:
        ex_graph = pickle.load(file)[4]
        N = ex_graph.num_nodes
else:
    ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
    adj = to_dense_adj(ex_graph.edge_index).squeeze().cuda()

    N = ex_graph.num_nodes
    ex_graph.edge_index = ex_graph.edge_index.to(device)
    ex_graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)

    EPS = 1e-12
    MAX_LENGTH = 128
    g = to_networkx(ex_graph, to_undirected=True, edge_attrs=['distance_m'])
    all_pairs = dict(nx.all_pairs_dijkstra(g, weight='distance_m'))
    delta_max = max(len(path) for target in all_pairs.values() for path in target[1].values())
    paths = torch.zeros((N, N, delta_max if delta_max < MAX_LENGTH else MAX_LENGTH)).long()
    dist = torch.full((N, N), float('inf'))
    for u, (lengths_u, paths_u) in all_pairs.items():
        for v, p in paths_u.items():
            dist[u, v] = lengths_u[v]
            p = (
                torch.tensor(p, device=device).long() if len(p) < MAX_LENGTH 
                else torch.full((MAX_LENGTH,), -1, device=device).long()
            )
            paths[u, v, 0:len(p)] = p
            paths[v, u, 0:len(p)] = p
    dist.fill_diagonal_(EPS)
    ex_graph.paths = paths.to(device)
    ex_graph.dist = dist.to(device)

# Show the shortest paths matrix of a node in the graph.
node1 = random.randint(0, N - 1)
node2 = random.randint(0, N - 1)
render_matrix(ex_graph.paths[node1, node2][None, :], sig_figs=0)

Matrix([[20, 19, 8, 9, 0, 0, 0, 0]])

In [11]:
# Feed the matrix to the GNN.
gnn.eval()
with torch.no_grad():
    out = gnn(ex_graph).to(device)

_, _, V = torch.pca_lowrank(out, q=10, center=True)
out = out - out.mean(dim=0)
render_matrix(out @ V)

Matrix([
[   2.53,  -0.589,  -0.497, -0.0249,  -0.0869,   -0.137, -0.0615,    -0.195,   -0.134,   -0.0508],
[   3.33,   0.318,  -0.402,   0.102,   -0.186,  -0.0296,  0.0167,    0.0613,   0.0624,     0.065],
[    3.5,   0.341,  -0.365,   0.135,    -0.17,  -0.0341,  0.0324,     0.127,    0.153,    0.0217],
[  -4.47,  -0.903,   -1.07,  -0.293,    0.256,    -0.22,  0.0443,   0.00612,    0.108,   -0.0378],
[   -3.6,  -0.965,  -0.116,    0.11,    0.147,   -0.348, -0.0142,    -0.095,  -0.0047,     0.179],
[  -4.14,   0.973,    -1.5,  -0.363,   -0.212,    0.176,  -0.127,    0.0329,  -0.0732,   -0.0537],
[  -2.37,  -0.454,   0.199,   0.445,  -0.0535,   -0.408,   -0.05,    0.0912,  0.00601,     -0.14],
[  0.366,   0.972,   0.255,    -0.4,   -0.226,   -0.247,  0.0375,     0.126,    -0.11,    0.0567],
[   2.97,   0.292,  -0.188,   0.102,    0.389,   0.0705,  -0.123,    0.0593,   0.0481,   0.00949],
[    3.2,  -0.238,    -0.2,   0.124,  -0.0208,   0.0508,  0.0578,   -0.0259,  -0.0239, -0.000419],
[

In [12]:
# Test out the Detector.
detector.eval()
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = detector(ex_graph, node1, node2).to(device)

print(node1, node2)
render_matrix(out.sigmoid())

7 11


Matrix([[0.45]])

#### Pre-Training of GNN on Edge Incidence

Next, we actually preprocess and train the GNN using the steps defined above.

In [13]:
# Init variables.
load_test_graphs = False

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)
test_dataset, _ = data.load_samples_by_graph(
    eval_path
)

# Configure the validation dataset.
EPS = 1e-12
MAX_LENGTH = 128
train_prop = 0.8
train_num = len(train_dataset)
train_keys = random.sample(list(train_dataset.keys()), k=int(train_num * train_prop))
val_dataset = {k: v for k, v in train_dataset.items() if k not in train_keys}
train_dataset = {k: v for k, v in train_dataset.items() if k in train_keys}

# Preprocess the data.
def generate_data(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        N = graph.num_nodes
        graph.edge_index = graph.edge_index.to(device)
        graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)

        # Distances and paths.
        g = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
        all_pairs = dict(nx.all_pairs_dijkstra(g, weight='distance_m'))
        delta_max = max(len(path) for target in all_pairs.values() for path in target[1].values())
        paths = torch.full((N, N, delta_max if delta_max < MAX_LENGTH else MAX_LENGTH), -1).long()
        dist = torch.full((N, N), float('inf'))
        for u, (lengths_u, paths_u) in all_pairs.items():
            for v, p in paths_u.items():
                dist[u, v] = lengths_u[v]
                p = (
                    torch.tensor(p, device=device) if len(p) < MAX_LENGTH 
                    else torch.full((MAX_LENGTH,), -1, device=device)
                )
                paths[u, v, 0:len(p)] = p
                paths[v, u, 0:len(p)] = p
        dist.fill_diagonal_(EPS)
        graph.paths = paths.to(device)
        graph.dist = dist.to(device)

        # Edges.
        combs = torch.triu_indices(N, N, offset=1, device=device)
        edge_codes = graph.edge_index[0] * N + graph.edge_index[1]
        existence = torch.isin(combs[0] * N + combs[1], edge_codes)
        graph.exclusion = combs[:, ~existence].to(device)
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), dim=0
        ).to(device)

    return graphs


def reshuffle(graphs):
    for graph in graphs:
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), dim=0
        ).to(device)

    return graphs


train_graphs = generate_data(train_dataset)
val_graphs = generate_data(val_dataset)

test_dataset = {k: v for k, v in test_dataset.items() if k not in ['eval_graph_unique_1000']}
if load_test_graphs:
    with open(save_path, 'rb') as file:
        test_graphs = pickle.load(file)
else:
    test_graphs = generate_data(test_dataset)
    with open(save_path, 'wb') as file:
        pickle.dump(test_graphs, file)

In [14]:
# Train the GNNEdgeDetector to reconstruct the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 150
es_patience = 5
train_edges = False

def test_loop_edges(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss, correct = 0, 0
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss += loss_fn(preds, graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds.sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, f1


def train_loop_edges(train_dataloader, val_dataloader, test_dataloader, model,
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('edge_detection', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        reshuffle(val_dataloader.dataset)
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss, _ = test_loop_edges(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = loss_fn(preds, graph.edges_y)

            # Backpropagation.
            (loss / batch_size).backward()
            model.invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Test the finished model.
    test_loop_edges(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


loss_fn = nn.BCEWithLogitsLoss()
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
if train_edges:
    optimizer = torch.optim.AdamW([
        {'params': detector.gnn.parameters(), 'lr': 3e-5},
        {'params': detector.classifier.parameters(), 'lr': 3e-4},
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_edges(train_dataloader, val_dataloader, test_dataloader, detector,
                     loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

In [15]:
if train_edges:
    torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
    torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
else:
    detector = torch.load('../outputs/e9_multistage_training/special/edge_detector_final.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/special/edge_detector_{model_type}_final.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence

We now test the trained model on the evaluation dataset. First, we we will render the output for clarity.

In [16]:
# Test out the Detector.
detector.eval().to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = detector(ex_graph, node1, node2)

print(node1, node2)
render_matrix(out.sigmoid())

9 18


Matrix([[0.000495]])

In [17]:
# Evaluate the GNN on its reconstruction of test graph adjacencies.
test_loop_edges(test_dataloader, detector, loss_fn)
pass

Test Error: 
 Accuracy: 92.7%, F1: 0.932 | P: 0.877 | R: 0.995 | Bal Acc: 92.8% | Avg loss: 0.219306 



### §2 Fine-tuning the GNN to Estimate Shortest-Path Distances

We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to estimate the distance of the shortest path between two given nodes in the graph. Such a model will assist the shortest-path prediction model, serving as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\bigg(\Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big);\, S, \mathcal{H}\bigg)$$
$$c_2(\Psi, \Psi) = \sqrt{2\operatorname{diag}(\Psi^2) - 2\Psi^2} \approx [SPD]$$
$$\mathbf{E} = \mathbb{E}\left[\frac{[SPD]_{ij}}{\delta(i, j)}\right]_{i, j \in [N]}$$

#### Model Definitions

We first define the model by attaching a simple GCN head to the GNN positional encoder.

In [18]:
# Define a class for shortest-path distance estimation and instantiate it.
class GNNShortestPathsEstimator(nn.Module):
    """
    Simple class to predict the shortest-path distance graphical lasso estimator (covariance).
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNShortestPathsEstimator, self).__init__()
        self.head = GCN(
            model_hparams['d_model'],
            model_hparams['d_model'],
            model_hparams['num_layers'],
            use_random_walk=True,
            skip_connection=True,
            dropout=model_hparams['dropout'],
            k=model_hparams['k_pe']
        )
        self.gate = nn.Parameter(torch.tensor(0.1))
        self.gnn = gnn

    def forward(self, graph: Data):
        graph = graph.clone()
        graph.x = self.gnn(graph)
        out = self.head(graph)
        out = torch.cdist(out, out, p=2)
        return self.gate * out

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the shortest-path distances matrix given a scene graph PyTorch `Data` object.

In [19]:
# Test out the SPD GNN.
spd_gnn = GNNShortestPathsEstimator(detector.gnn).eval()
with torch.no_grad():
    out = spd_gnn(ex_graph).to(device)

render_matrix(out)

Matrix([
[   0,   1.54,  1.74,    2.0,  1.89,  1.59,    2.03,   1.82, 2.18,  1.48,  1.61,    1.83,   1.18,  1.64,  1.42,    1.62,  2.11,    2.57,   2.2,  2.06,   1.99,    2.52,  2.04,   2.02,  2.12,  1.81,  1.97,  1.93,   2.0,  2.03,  2.29,  2.56],
[1.54, 0.0011, 0.823,   2.49,  2.32,  1.92,    2.34,   1.39, 1.93,  1.55,  1.27,    1.31,   1.71,   1.9,  1.83,    1.42,  2.18,    2.55,  2.35,  2.46,    2.2,    2.72,  2.28,   2.29,  2.23,  1.76,  1.82,  1.76,  1.76,  1.99,  2.38,  2.65],
[1.74,  0.823,     0,   2.54,   2.4,  2.01,    2.39,   1.54,  1.8,  1.67,  1.41,    1.25,   1.86,  2.02,  1.94,    1.45,  2.22,    2.49,  2.36,  2.49,   2.24,    2.69,  2.31,   2.36,   2.3,  1.81,  1.86,  1.81,   1.8,  2.03,  2.38,  2.66],
[ 2.0,   2.49,  2.54, 0.0011, 0.637,  1.21,   0.742,   2.31, 2.58,  2.39,  2.54,    2.54,   2.31,  2.45,  2.34,    2.45,  1.37,    1.91,  1.15, 0.936,   1.05,    1.54, 0.999,   0.98,   2.3,  2.25,   2.3,  2.34,  2.33,  2.48,  2.63,  2.47],
[1.89,   2.32,   2.4,  0.637,  

In [20]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    34.9,    32.1,   111.0,    99.3,   129.0,   110.0,   158.0,     2.8,    40.3,    60.6,    35.0,    41.2,    31.8,    30.9,    44.4,   109.0,   113.0,   110.0,    87.3,    94.8,   124.0,   115.0,   106.0,   154.0,   127.0,   132.0,   138.0,   154.0,   112.0,   131.0,   156.0],
[   34.9, 1.0e-12,    27.3,   140.0,   129.0,   159.0,   139.0,   129.0,    32.1,    40.5,    31.9,    22.0,    45.4,    3.08,    26.1,    25.9,   138.0,   142.0,   139.0,   117.0,   124.0,   153.0,   144.0,   136.0,   126.0,    98.3,   103.0,   109.0,   139.0,    83.7,   122.0,   127.0],
[   32.1,    27.3, 1.0e-12,   138.0,   126.0,   156.0,   137.0,   150.0,    29.3,    39.7,    53.0,    5.28,    35.1,    24.2,    1.17,    15.5,   136.0,   140.0,   136.0,   114.0,   121.0,   150.0,   142.0,   133.0,   147.0,   119.0,   125.0,   130.0,   142.0,   105.0,   118.0,   148.0],
[  111.0,   140.0,   138.0, 1.0e-12,    35.7,    65.7,    46.6,   152.0,   108.0,   146.0,   166.0,   141.0,   147.0,   1

In [21]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[     0,  0.044, 0.0543,  0.018,  0.019, 0.0123,  0.0184, 0.0116,  0.777, 0.0368, 0.0265,  0.0522, 0.0286, 0.0515, 0.0461,  0.0365, 0.0193,  0.0227, 0.0201, 0.0236,  0.021,  0.0204, 0.0177,  0.019, 0.0137, 0.0143,  0.0149,  0.014,   0.013,  0.018, 0.0175, 0.0164],
[ 0.044, 1.1e+9, 0.0302, 0.0177,  0.018, 0.0121,  0.0168, 0.0108,   0.06, 0.0381, 0.0399,  0.0596, 0.0377,  0.618, 0.0702,  0.0549, 0.0158,  0.0179, 0.0169, 0.0211, 0.0177,  0.0178, 0.0158, 0.0169, 0.0177, 0.0179,  0.0176, 0.0161,  0.0127, 0.0237, 0.0195, 0.0208],
[0.0543, 0.0302,      0, 0.0185, 0.0191, 0.0129,  0.0175, 0.0103, 0.0614,  0.042, 0.0267,   0.237,  0.053, 0.0835,   1.66,  0.0938, 0.0164,  0.0178, 0.0173, 0.0219, 0.0185,  0.0179, 0.0163, 0.0177, 0.0157, 0.0151,  0.0149, 0.0139,  0.0127, 0.0194, 0.0202, 0.0179],
[ 0.018, 0.0177, 0.0185, 1.1e+9, 0.0179, 0.0185,  0.0159, 0.0152, 0.0238, 0.0164, 0.0153,  0.0181, 0.0157, 0.0178, 0.0171,  0.0163, 0.0302,  0.0386,  0.763, 0.0394, 0.0336,  0.0258, 0.0194,  0.023

#### Definition of a Custom Loss Function: Graphical Lasso Estimator 

We seek to reproduce the [Graphical Lasso Estimator](https://en.wikipedia.org/wiki/Graphical_lasso) custom loss function within the PyTorch framework. Since such an error and gradient computation function requires a differentiable interpretation of the $L_1$ regularization penalty, we must define a new subclass of `torch.autograd.Function` to implement this regression objective within the working environment.

The Graphical Lasso Estimator is defined through the following mathematical optimizer:

$$\hat{\Theta} = \argmax_{\Theta \succ 0} L(\Theta) = \argmax_{\Theta \succ 0}\left(\log\det(\Theta) - \operatorname{tr}(S\Theta) - \lambda\sum_{i, j}|\Theta_{ij}|\right)$$

Thus, it has the following derivative evaluation:

$$\nabla_{\Theta} L(\Theta) = \frac{1}{\det(\Theta)} \det(\Theta) \Theta^{-\top} - S^T - \lambda \begin{cases}1 & \text{if } \Theta_{ij} > 0 \\ 0 & \text{if } \Theta_{ij} = 0 \\ -1 & \text{if } \Theta_{ij} < 0\end{cases}$$
$$\nabla_{\Theta} L(\Theta) = \Theta^{-1} - S - \lambda \operatorname{sign}(\Theta)$$

In [22]:
LAMBDA = 1e-7


class GraphicalLassoEstimator(torch.autograd.Function):
    @staticmethod
    def forward(ctx, preds, targets):
        """
        Computes the loss value for the Graphical Lasso Estimator loss function.
        """
        ctx.save_for_backward(preds, targets)
        sign, logdet = torch.linalg.slogdet(preds)
        assert (sign > 0).all()
        return -logdet + torch.trace(targets @ preds) - LAMBDA * preds.abs().sum()

    @staticmethod
    def backward(ctx, grad_output):
        """
        Computes custom gradients with respect to the inputs. Honors the requirement
        for L1 differentiability within the PyTorch framework.
        """
        preds, targets = ctx.saved_tensors
        grad_predictions = grad_targets = None
        if ctx.needs_input_grad[0]:
            grad_predictions = grad_output * - (preds.inverse() - targets - LAMBDA * preds.sign())
        if ctx.needs_input_grad[1]:
            grad_targets = grad_output * preds
        return grad_predictions, grad_targets

#### Fine-Tuning of GNN on Shortest-Path Distances

We preprocess and train the GNN using the steps defined above.

In [23]:
# Train the GNN to reconstruct the shortest-path distances of the graph.
batch_size = 4
val_freq = 5
epochs = 200
es_patience = 5
train_dists = False

def test_loop_dists(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model['dists'].to(device).eval()
    model['edges'].to(device).eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss = {'dists': 0, 'edges': 0}
    correct, error_norm = 0, 0
    tp = fp = fn = tn = 0
    
    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = {
                'dists': model['dists'](graph),
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }

            test_loss['dists'] += loss_fn['dists'](preds['dists'], graph.dist).item()
            error = preds['dists'] / graph.dist
            error.fill_diagonal_(0)
            error_norm += torch.linalg.matrix_norm(error) / error.shape[0]

            test_loss['edges'] += loss_fn['edges'](preds['edges'], graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds['edges'] > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds['edges'].sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss['dists'] /= size
    error_norm /= size
    print(f"Test Error #1: \n Avg error: {error_norm:>0.3f} \n Avg loss: {test_loss['dists']:>8f} \n")

    test_loss['edges'] /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error #2: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss['edges']:>8f} \n")

    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss_dists': test_loss['dists'],
            f'{wandb_prefix}/error_norm': error_norm,
            f'{wandb_prefix}/loss_edges': test_loss['edges'],
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, error_norm


def train_loop_dists(train_dataloader, val_dataloader, test_dataloader, model,
                     loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model['dists'].to(device).train()
    model['edges'].to(device).train()
    val_loss: float = 0
    best_val, best_state, bad_runs, best_state = float('inf'), None, 0, {}
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('shortest_path_distances', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn['dists']),
        **loss_hparams(loss_fn['edges']),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss, _ = test_loop_dists(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss['dists'])
            if (val_loss['dists'] < best_val['edges'] - 1e-3 
                    and val_loss['edges'] < best_val['edges'] - 1e-3):
                best_val, bad_runs = val_loss, 0
                best_state['dists'] = copy.deepcopy(model['dists'].state_dict())
                best_state['edges'] = copy.deepcopy(model['edges'].state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model['dists'].train()
            model['edges'].train()
        
        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = {
                'dists': model['dists'](graph),
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }
            preds['dists'] = preds['dists']
            loss = {
                'dists': loss_fn['dists'](preds['dists'], graph.dist),
                'edges': loss_fn['edges'](preds['edges'], graph.edges_y)
            }

            # Backpropagation.
            loss['dists'] /= graph.num_nodes
            ((loss['dists'] + loss['edges']) / batch_size).backward()
            model['edges'].invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model['dists'].parameters(), max_norm=1.0)
                clip_grad_norm_(model['edges'].parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                current = j
                wandb.log({
                    'train/loss_dists': loss['dists'],
                    'train/loss_edges': loss['edges'],
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss #1: {loss['dists'].item():>7f}  [{current:>5d}/{size:>5d}]")
                print(f"Loss #2: {loss['edges'].item():>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state:
        model['dists'].load_state_dict(best_state['dists'])
        model['edges'].load_state_dict(best_state['edges'])

    # Test the finished model.
    test_loop_dists(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish MSE/Graphical-Lasso loss.
loss_fn = {
    'dists': nn.MSELoss(),
    'edges': nn.BCEWithLogitsLoss()
}
if train_dists:
    optimizer = torch.optim.AdamW([
        {'params': gnn.parameters(), 'lr': 3e-5},
        {'params': detector.classifier.parameters(), 'lr': 3e-4},
        {'params': spd_gnn.head.parameters(), 'lr': 3e-4},
        {'params': [spd_gnn.gate], 'lr': 3e-4}
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_dists(train_dataloader, val_dataloader, test_dataloader, 
                    {'dists': spd_gnn, 'edges': detector}, loss_fn, optimizer, 
                    scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

In [24]:
if train_dists:
    torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
    torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
else:
    detector = torch.load('../outputs/e9_multistage_training/suite1/edge_detector.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/suite1/edge_detector_{model_type}.pt'))

In [25]:
if train_dists:
    torch.save(spd_gnn, '../outputs/e9_multistage_training/spd_gnn.pt')
    torch.save(spd_gnn.gnn.state_dict(), f'../outputs/e9_multistage_training/spd_gnn_{model_type}.pt')
else:
    spd_gnn = torch.load(f'../outputs/e9_multistage_training/suite1/spd_gnn.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/suite1/spd_gnn_{model_type}.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence and Shortest-Paths Distance Estimation

We test the pre-trained model on the evaluation dataset. We we will render the output error matrix $\mathbf{E}$ for visibility.

In [26]:
# Test out the SPD GNN.
spd_gnn.eval().to(device)
with torch.no_grad():
    out = spd_gnn(ex_graph)

render_matrix(out)

Matrix([
[    0,  28.1,  34.3, 112.0, 131.0,  109.0, 127.0,  49.2,  25.6,  34.3,  29.4,  44.1,  33.8,  36.4,  40.5,   43.0, 142.0, 144.0, 136.0, 158.0, 153.0, 114.0,  151.0, 150.0,   52.6,  38.9,   45.1,   50.8,  49.4,  42.2,  47.7,  46.6],
[ 28.1,     0,  11.1, 134.0, 152.0,  132.0, 148.0,  52.7,  18.3,  17.1,  13.7,  22.7,  18.1,  21.1,  23.5,   26.0, 164.0, 164.0, 158.0, 177.0, 174.0, 136.0,  170.0, 170.0,   55.2,  44.3,   49.0,   54.3,  51.6,  49.6,  52.4,  51.8],
[ 34.3,  11.1,     0, 136.0, 154.0,  134.0, 149.0,  48.8,  26.3,  21.0,  22.0,  20.0,  21.2,  19.9,  19.0,   24.0, 166.0, 165.0, 161.0, 179.0, 176.0, 138.0,  171.0, 171.0,   51.4,  42.6,   46.8,   51.6,  48.7,  48.2,  49.9,  49.6],
[112.0, 134.0, 136.0, 0.108,  22.2,   11.2,  23.9,  97.7, 134.0, 133.0, 132.0, 139.0, 130.0, 134.0, 134.0,  138.0,  34.2,  48.2,  29.7,  63.6,  50.1,  15.4,   58.0,  50.7,  104.0,  99.0,   98.7,   99.4, 100.0,  95.0,  97.6,  96.6],
[131.0, 152.0, 154.0,  22.2,     0,   30.5,  9.77, 114.0, 152.0

In [27]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    34.9,    32.1,   111.0,    99.3,   129.0,   110.0,   158.0,     2.8,    40.3,    60.6,    35.0,    41.2,    31.8,    30.9,    44.4,   109.0,   113.0,   110.0,    87.3,    94.8,   124.0,   115.0,   106.0,   154.0,   127.0,   132.0,   138.0,   154.0,   112.0,   131.0,   156.0],
[   34.9, 1.0e-12,    27.3,   140.0,   129.0,   159.0,   139.0,   129.0,    32.1,    40.5,    31.9,    22.0,    45.4,    3.08,    26.1,    25.9,   138.0,   142.0,   139.0,   117.0,   124.0,   153.0,   144.0,   136.0,   126.0,    98.3,   103.0,   109.0,   139.0,    83.7,   122.0,   127.0],
[   32.1,    27.3, 1.0e-12,   138.0,   126.0,   156.0,   137.0,   150.0,    29.3,    39.7,    53.0,    5.28,    35.1,    24.2,    1.17,    15.5,   136.0,   140.0,   136.0,   114.0,   121.0,   150.0,   142.0,   133.0,   147.0,   119.0,   125.0,   130.0,   142.0,   105.0,   118.0,   148.0],
[  111.0,   140.0,   138.0, 1.0e-12,    35.7,    65.7,    46.6,   152.0,   108.0,   146.0,   166.0,   141.0,   147.0,   1

In [28]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[    0, 0.805,  1.07,     1.01,  1.31,    0.842,  1.16, 0.312,  9.14, 0.849, 0.485,  1.26,  0.82,  1.15,  1.31,    0.968,  1.31,  1.28,  1.24,  1.81,  1.61,  0.92,     1.32,  1.41,    0.341, 0.306,    0.341,    0.368,  0.32, 0.375,  0.365,   0.299],
[0.805,     0, 0.407,    0.956,  1.18,    0.832,  1.06, 0.409, 0.568, 0.421, 0.431,  1.03, 0.399,  6.84, 0.901,      1.0,  1.19,  1.15,  1.14,  1.52,   1.4, 0.889,     1.18,  1.25,    0.439, 0.451,    0.474,    0.497, 0.372, 0.593,  0.429,   0.407],
[ 1.07, 0.407,     0,    0.987,  1.22,     0.86,  1.09, 0.326, 0.897,  0.53, 0.416,  3.79, 0.604, 0.825,  16.2,     1.55,  1.22,  1.19,  1.18,  1.57,  1.46, 0.921,     1.21,  1.29,    0.351, 0.357,    0.376,    0.395, 0.344,  0.46,  0.422,   0.334],
[ 1.01, 0.956, 0.987, 1.08e+11, 0.623,    0.171, 0.512, 0.644,  1.23, 0.909, 0.795, 0.989, 0.887, 0.973, 0.986,    0.923, 0.751, 0.974,  19.6,  2.68,  1.61, 0.256,     1.13,  1.19,    0.699, 0.637,    0.658,    0.668, 0.644,  0.56,  0.588,  

In [29]:
def are_models_equal(model1, model2):
    # 1. Check if both models have the exact same state_dict keys
    if model1.state_dict().keys() != model2.state_dict().keys():
        return False
    
    # 2. Check if all parameters and buffers are exactly equal
    for key, value1 in model1.state_dict().items():
        value2 = model2.state_dict()[key]
        
        # Use torch.equal for strict element-wise and structural equality
        if not torch.equal(value1, value2):
            return False
            
    return True

are_models_equal(detector.gnn, spd_gnn.gnn)

True

In [30]:
# Evaluate the GNN on its reconstruction of test graph edge incidences and shortest-path distances together.
models = {'dists': spd_gnn, 'edges': detector}
test_loop_dists(test_dataloader, models, loss_fn)
pass

Test Error #1: 
 Avg error: 2.700 
 Avg loss: 1313.390494 

Test Error #2: 
 Accuracy: 93.4%, F1: 0.937 | P: 0.890 | R: 0.990 | Bal Acc: 93.4% | Avg loss: 0.268602 



### §3 Fine-tuning the GNN to Predict Shortest-Path Subgraph Adjacencies

We now wish to optimize the jointly fine-tuned GNN (R-PEARL or Graph Transformer) to predict the shortest path itself between two given nodes in the graph. Such a model will serve as the actual backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H}\,)\big];\, \mathcal{T}\Big)$$
$$\mathbf{X} = \left[\mathbf{x}_i \sim \text{Cauchy}(0,\, \mathbf{I})\right]_{i \in [N]}^\top \in \mathbb{R}^{N \times D} \qquad D \gg N$$
$$\forall\, t_i \in T \qquad \mathbf{x}_i \sim \text{Cauchy}(i,\, \mathbf{I}) \implies \mathbf{X}(T) = \left[\mathbf{x}_t \sim \text{Cauchy}(t,\, \mathbf{I})\right]^\top_{t \in T}$$
$$\forall\, u, v \in V^2 \quad u \rightsquigarrow v \qquad \mathbf{x}_u \sim \text{Cauchy}(1,\, \mathbf{I}) \qquad \mathbf{x}_v \sim \text{Cauchy}\big(\delta(u, v),\, \mathbf{I}\big)$$
$$\qquad \hat{U}_{1} = u \in V \qquad \hat{U}_{t+1} = \Phi\Big(\mathbf{X}\big(U_{1:t}\big) + \mathbf{\Psi};\, \mathcal{T}\Big) \in V^{t + 1} \qquad \hat{U}_{1:T} = (u,\, \cdots, v) = \hat{U}(u, v) \in V^T$$
$$\mathbf{E} = \mathbb{E}\left[\frac{|\hat{U}(u, v)|}{\delta(u, v)}\right]_{u, v \in V^2}

#### Model Definitions
We first define the model by attaching a full Autoregressive Graph Transformer (AGT) to the GNN positional encoder.

In [31]:
# Define a class for edge detection and instantiate it.
class GNNShortestPathNavigator(GNNEdgeDetector):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer], max_length=128):
        super().__init__(gnn)
        self.MAX_LENGTH = max_length
        self.shape = gnn.out_features
        self.head = SemanticGraphTransformer(
            node_feature_dim=model_hparams['d_model'],
            num_layers=model_hparams['num_layers'],
            d_model=model_hparams['d_model'],
            heads=model_hparams['heads'],
            dropout=model_hparams['dropout'],
            k_gt=model_hparams['k_gt'],
        )
        self.classifier = nn.Linear(in_features=self.shape, out_features=1)
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, self.shape))

    def forward(self, graph: Data):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        
        feed_graph = self.graph.clone()
        feed_graph.x = graph.x + self.cached_pe
        return self.classifier(self.head(feed_graph))
    
    def generate(self, graph: Data, node1: int, node2: int):
        """Autoregressively generates a path from Node 1 to Node 2."""
        
        # Establish Cauchy distribution.
        N, D = graph.num_nodes, self.shape
        graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, D)).to(device)

        # Set up variables.
        count = 1.0
        history = []
        preds = [node1]
        complete = list(range(N))

        # Run generation loop.
        while not (preds[-1] == node2 or history == complete 
                or len(preds) > self.MAX_LENGTH):
            graph.x[preds[-1]] = Cauchy(loc=count, scale=1.0).sample((1, D)).to(device)
            preds.append(self(graph).argmax(dim=0).item())
            bisect.insort(history, preds[-1])
            count += 1.0
        
        # Clean up and return.
        graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)
        return torch.tensor([preds], device=device).T
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
navigator = GNNShortestPathNavigator(spd_gnn.gnn)

In [32]:
# Test out the Navigator.
navigator.eval().to(device)
with torch.no_grad():
    out = navigator(ex_graph)

render_matrix(out.softmax(dim=0).T)

Matrix([[0.0218, 0.0201, 0.0196, 0.0293, 0.021, 0.0257, 0.0214, 0.0286, 0.0335, 0.0337, 0.0254, 0.0328, 0.0319, 0.0247, 0.0355, 0.0318, 0.0176, 0.0329, 0.0224, 0.0339, 0.0345, 0.037, 0.0358, 0.0332, 0.0295, 0.0464, 0.0389, 0.0463, 0.0339, 0.0425, 0.0418, 0.0365]])

In [33]:
# Test out the Navigator's generation abilities.
N = ex_graph.num_nodes
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = navigator.generate(ex_graph, node1, node2)

print(node1, node2)
print(ex_graph.paths[node1, node2].tolist())
render_matrix(out.T, sig_figs=0)

27 7
[27, 24, 7, 0, 0, 0, 0, 0]


Matrix([[27, 10, 0, 2, 2, 0, 23, 24, 24, 24, 25, 18, 19, 0, 9, 31, 24, 2, 10, 30, 17, 28, 29, 10, 27, 10, 14, 29, 29, 16, 23, 23, 23, 23, 30, 30, 29, 24, 24, 25, 24, 24, 14, 15, 11, 15, 9, 11, 1, 26, 26, 25, 25, 26, 24, 24, 24, 24, 26, 26, 29, 29, 29, 29, 29, 29, 24, 24, 24, 24, 24, 25, 25, 25, 25, 25, 25, 25, 25, 24, 25, 25, 25, 25, 25, 17, 22, 26, 29, 27, 29, 27, 22, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 29, 29, 29, 27, 27, 27, 25, 26, 26, 26, 26, 26, 26, 26, 26, 26, 21, 21, 21, 21, 1, 29]])

#### Fine-Tuning of GNN on Shortest Paths
Finally, we preprocess and train the GNN using the steps defined above.

In [ ]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
batch_size = 4
val_freq = 1 # 5
epochs = 1 # 200
es_patience = 5
batch_prop = 1.0
detour_bce = False
train_paths = True

def test_loop_paths(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model['paths'].to(device).eval()
    model['edges'].to(device).eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss = {'paths': 0, 'edges': 0}
    correct = {'paths': 0, 'edges': 0}
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            
            # Edges.
            preds = {
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }
            test_loss['edges'] += loss_fn['edges'](preds['edges'], graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds['edges'] > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct['edges'] += (
                (preds['edges'].sigmoid() > 0.5).float() == graph.edges_y
            ).float().mean().item()

            # Paths.
            N, D = graph.num_nodes, model['paths'].gnn.out_features
            batch_len = int(batch_prop * N)
            batch = torch.randperm(N)[:batch_len]
            batch_u = batch[:batch_len // 2].tolist()
            batch_v = batch[batch_len // 2:].tolist()
            batch_uv = list(zip(batch_u, batch_v))
            
            graph_loss = 0
            for u, v in batch_uv:
                # Set up variables.
                count = 1.0
                agt_preds = [u]
                out, history = [], []
                complete = list(range(N))
                ground_truth = graph.paths[u, v, 1:]
                ground_truth = ground_truth[ground_truth >= 0]
                graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, D)).to(device)

                # Run generation loop with modified constraints.
                while not (agt_preds[-1] == v or history == complete 
                        or len(agt_preds) > ground_truth.shape[0]):
                    graph.x[agt_preds[-1]] = Cauchy(loc=count, scale=1.0).sample((1, D)).to(device)
                    logits = model['paths'](graph).T
                    agt_preds.append(logits.argmax(dim=1).item())
                    bisect.insort(history, agt_preds[-1])
                    out.append(logits)
                    count += 1.0

                if not out:
                    continue
                out = torch.cat(out).to(device)
                graph_loss += loss_fn['paths'](
                    out, ground_truth[:out.shape[0]]
                )
                preds['paths'] = agt_preds

            test_loss['paths'] += graph_loss.item() / max(len(batch_uv), 1)
            graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, D)).to(device)

    test_loss['paths'] /= size
    print(f"Test Error #1: \n Avg loss: {test_loss['paths']:>8f} \n")

    test_loss['edges'] /= size
    correct['edges'] /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error #2: \n Accuracy: {(100*correct['edges']):>0.1f}%, F1: {f1:.3f} "
          f"| P: {precision:.3f} | R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% "
          f"| Avg loss: {test_loss['edges']:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss_paths': test_loss['paths'],
            f'{wandb_prefix}/loss_edges': test_loss['edges'],
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss


def train_loop_paths(train_dataloader, val_dataloader, test_dataloader, model, 
                     loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model['paths'].train().to(device)
    model['edges'].train().to(device)
    best_state, bad_runs = {}, 0
    best_val = {'paths': float('inf'), 'edges': float('inf')}
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('path_navigation', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'detour_bce': detour_bce,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss = test_loop_paths(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss['paths'] + val_loss['edges'])
            if (val_loss['paths'] < best_val['paths'] - 1e-3 
                    or val_loss['edges'] < best_val['edges'] - 1e-3):
                bad_runs = 0
                best_val = val_loss.copy()
                best_state['paths'] = copy.deepcopy(model['paths'].state_dict())
                best_state['edges'] = copy.deepcopy(model['edges'].state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val #1 "
                          f"{best_val['paths']:>8f}, best val #2 "
                          f"{best_val['edges']:>8f})")
                    break
            model['paths'].train()
            model['edges'].train()
        
        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            graph = train_dataloader.dataset[idx]

            # Set up randomized paths for training.
            N, D = graph.num_nodes, model['paths'].gnn.out_features
            batch_len = int(batch_prop * N)
            batch = torch.randperm(N)[:batch_len]
            batch_u = batch[:batch_len // 2].tolist()
            batch_v = batch[batch_len // 2:].tolist()
            batch_uv = list(zip(batch_u, batch_v))

            graph_loss = 0
            for u, v in batch_uv:
                # Set up variables.
                count = 1.0
                agt_preds = [u]
                out, history = [], []
                complete = list(range(N))
                ground_truth = graph.paths[u, v]
                ground_truth = ground_truth[ground_truth >= 0]

                # Run generation loop with modified constraints.
                graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, D)).to(device)
                while not (agt_preds[-1] == v or history == complete 
                        or len(agt_preds) > ground_truth.shape[0] - 1):
                    graph.x[ground_truth[int(count)-1]] = Cauchy(
                        loc=count, scale=1.0).sample((1, D)
                    ).to(device)
                    logits = model['paths'](graph).T
                    agt_preds.append(logits.argmax(dim=1).item())
                    bisect.insort(history, agt_preds[-1])
                    out.append(logits)
                    count += 1.0

                # Compute Cross-Entropy loss on full path.
                out = torch.cat(out).to(device)
                graph_loss += loss_fn['paths'](
                    out, ground_truth[1:out.shape[0] + 1]
                )
            
            # Get edge predictions and losses.
            preds_edges = torch.stack([
                model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = {
                'paths': graph_loss / max(len(batch_uv), 1),
                'edges': loss_fn['edges'](preds_edges, graph.edges_y)
            }

            # Backpropagation.
            ((loss['paths'] + loss['edges']) / batch_size).backward()
            model['edges'].invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model['paths'].parameters(), max_norm=1.0)
                clip_grad_norm_(model['edges'].parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                current = j
                wandb.log({
                    'train/loss_paths': loss['paths'],
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss #1: {loss['paths'].item():>7f}  [{current:>5d}/{size:>5d}]")
                print(f"Loss #2: {loss['edges'].item():>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state:
        model['paths'].load_state_dict(best_state['paths'])
        model['edges'].load_state_dict(best_state['edges'])

    # Test the finished model.
    test_loop_paths(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish Cross-Entropy Loss for autoregressive path generation.
loss_fn = {'paths': nn.CrossEntropyLoss(), 'edges': nn.BCEWithLogitsLoss()}
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
if train_paths:
    optimizer = torch.optim.AdamW([
        {'params': navigator.gnn.parameters(), 'lr': 3e-5},
        {'params': navigator.head.parameters(), 'lr': 3e-5},
        {'params': navigator.classifier.parameters(), 'lr': 3e-4},
        {'params': detector.classifier.parameters(), 'lr': 3e-4}
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_paths(train_dataloader, val_dataloader, test_dataloader, 
                     {'paths': navigator, 'edges': detector}, loss_fn, optimizer, 
                     scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/arushar/.netrc.
wandb: Currently logged in as: arushar (alelab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Validation #1
Test Error #1: 
 Avg loss: 3.525080 

Test Error #2: 
 Accuracy: 99.4%, F1: 0.994 | P: 0.989 | R: 1.000 | Bal Acc: 99.4% | Avg loss: 0.013578 

Epoch #1
Loss #1: 3.570501  [    3/   32]
Loss #2: 0.191611  [    3/   32]
Loss #1: 3.588577  [    7/   32]
Loss #2: 0.062144  [    7/   32]
Loss #1: 3.705897  [   11/   32]
Loss #2: 0.008538  [   11/   32]
Loss #1: 3.452922  [   15/   32]
Loss #2: 0.076202  [   15/   32]
Loss #1: 3.714905  [   19/   32]
Loss #2: 0.028122  [   19/   32]
Loss #1: 3.917129  [   23/   32]
Loss #2: 0.040840  [   23/   32]
Loss #1: 3.671576  [   27/   32]
Loss #2: 0.047463  [   27/   32]
Loss #1: 3.889946  [   31/   32]
Loss #2: 0.016521  [   31/   32]
Test Error #1: 
 Avg loss: 3.483839 

Test Error #2: 
 Accuracy: 93.9%, F1: 0.941 | P: 0.902 | R: 0.984 | Bal Acc: 93.9% | Avg loss: 0.238648 



epoch,▁▁▁▁▁▁▁▁▁
global_step,▁▂▃▄▅▆▇█
test/bal_acc,▁
test/f1,▁
test/loss_edges,▁
test/loss_paths,▁
test/precision,▁
test/recall,▁
train/loss_paths,▃▃▅▁▅█▄█
train/lr,▁▁▁▁▁▁▁▁
+6,...


In [35]:
if train_paths:
    torch.save(navigator, '../outputs/e9_multistage_training/path_navigator.pt')
    torch.save(navigator.gnn.state_dict(), f'../outputs/e9_multistage_training/path_navigator_{model_type}.pt')
else:
    pass
    # navigator = torch.load('../outputs/e9_multistage_training/suite1/path_navigator.pt', weights_only=False)
    # gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/suite1/path_navigator_{model_type}.pt'))

#### Evaluation of Fine-Tuned GNN on Shortest Paths
We thus test the fine-tuned model on the evaluation dataset. First, we we will render the output for clarity.

In [36]:
# Test out the Navigator.
navigator.eval().to(device)
with torch.no_grad():
    out = navigator(ex_graph)

render_matrix(out.softmax(dim=0).T)

Matrix([[0.034, 0.0284, 0.0239, 0.0309, 0.0239, 0.0251, 0.0215, 0.0366, 0.0566, 0.0214, 0.0232, 0.0158, 0.0133, 0.0481, 0.0193, 0.0423, 0.0183, 0.0479, 0.0274, 0.0527, 0.0307, 0.0367, 0.0344, 0.0255, 0.025, 0.049, 0.031, 0.0392, 0.0136, 0.0226, 0.0302, 0.0514]])

In [37]:
# Test out the Navigator's generation abilities.
N = ex_graph.num_nodes
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = navigator.generate(ex_graph, node1, node2)

print(node1, node2)
print(ex_graph.paths[node1, node2].tolist())
render_matrix(out.T, sig_figs=0)

0 3
[3, 18, 19, 8, 0, 0, 0, 0]


Matrix([[0, 16, 2, 31, 10, 13, 11, 11, 10, 10, 6, 14, 28, 2, 2, 13, 13, 31, 12, 9, 17, 23, 28, 2, 2, 2, 2, 0, 0, 0, 10, 9, 10, 10, 9, 9, 10, 15, 15, 11, 27, 11, 11, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 20, 20, 30, 30, 25, 25, 25, 25, 25, 25, 17, 24, 24, 24, 24, 24, 24, 24, 24, 25, 25, 13, 13, 13, 11, 10, 10, 14, 25, 10, 10, 10, 10, 10, 10, 10, 14, 14, 14, 14, 27, 27, 27, 14, 27, 27, 27, 27, 27, 27, 27, 11, 25, 10, 10, 10, 10, 10, 10, 25, 10, 10, 14, 14, 14, 1, 1, 1, 1, 1, 1, 14, 1, 1]])

In [38]:
# Evaluate the GNN on its reconstruction of test graph shortest paths.
test_loop_paths(test_dataloader, {'paths': navigator, 'edges': detector}, loss_fn)
pass

Test Error #1: 
 Avg loss: 3.480433 

Test Error #2: 
 Accuracy: 93.6%, F1: 0.939 | P: 0.894 | R: 0.988 | Bal Acc: 93.6% | Avg loss: 0.244080 



In [39]:
# Evaluate the GNN on its reconstruction of test graph shortest path distances.
spd_gnn.gnn.load_state_dict(navigator.gnn.state_dict())
test_loop_dists(
    test_dataloader, {'dists': spd_gnn, 'edges': detector},
    {'dists': nn.MSELoss(), 'edges': nn.BCEWithLogitsLoss()}
)
pass

Test Error #1: 
 Avg error: 2.710 
 Avg loss: 1372.092908 

Test Error #2: 
 Accuracy: 93.4%, F1: 0.937 | P: 0.886 | R: 0.994 | Bal Acc: 93.3% | Avg loss: 0.265511 

